# Trace-Bench M3 — UI Launch + Browse + Resume + TensorBoard/MLflow

This notebook demonstrates M3 UI behavior:
- Launch a small stub run (2×2 matrix) to produce run artifacts
- Start the Gradio UI with `trace-bench ui`
- Browse runs, filter results, inspect jobs
- Resume a previous run using the UI resume flow
- TensorBoard and MLflow integration (optional)

**Artifacts under `runs/<run_id>/` are canonical.** The UI reads filesystem; MLflow/TB mirror it.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/guru-code-expert/Trace-Bench/blob/m3/deliverable/notebooks/03_ui_launch_monitor.ipynb)

In [ ]:
from datetime import date
from pathlib import Path
import os

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

def bench_dir(project="bench", sub="trace_bench_m3", local="/content/bench"):
    drive_root = Path("/content/drive/MyDrive")
    root = drive_root if drive_root.is_dir() else Path(local)
    out = root / project / date.today().isoformat() / sub
    out.mkdir(parents=True, exist_ok=True)
    return str(out)

RUNS_DIR = bench_dir()
os.environ["RUNS_DIR"] = RUNS_DIR
print("Runs dir:", RUNS_DIR)


In [ ]:
# Clone repos + install deps
!git clone --depth 1 --branch m3/deliverable https://github.com/guru-code-expert/Trace-Bench.git 2>/dev/null || (cd Trace-Bench && git pull)
!git clone --depth 1 --branch experimental https://github.com/guru-code-expert/OpenTrace.git 2>/dev/null || (cd OpenTrace && git pull)
%cd Trace-Bench
!python -m pip install -q pyyaml gradio
!python -m pip install -q -e /content/OpenTrace
!PYTHONPATH=/content/OpenTrace:/content/Trace-Bench:$PYTHONPATH python -c "import opto, trace_bench; print('imports ok')"


In [ ]:
%%bash
set -euo pipefail
cd /content/Trace-Bench

# Use the bundled M3 demo config
PYTHONPATH=/content/OpenTrace:/content/Trace-Bench:$PYTHONPATH python -m trace_bench run --config configs/m3_ui_demo.yaml --runs-dir "$RUNS_DIR"

echo ""
echo "--- Run artifacts ---"
ls -la "$RUNS_DIR"/*/
echo ""
echo "--- results.csv (head) ---"
head -5 "$RUNS_DIR"/*/results.csv 2>/dev/null || echo "(no results.csv found)"


In [ ]:
# Show run summary
import json, glob

for summary_path in sorted(glob.glob(f"{RUNS_DIR}/*/summary.json")):
    with open(summary_path) as f:
        summary = json.load(f)
    run_id = summary_path.split("/")[-2]
    print(f"Run: {run_id}")
    print(f"  Total jobs: {summary.get('total_jobs', '?')}")
    print(f"  Counts: {summary.get('counts', {})}")
    print()

## Launch Gradio UI

The UI has 3 tabs:
1. **Launch Run** — discover tasks/trainers dynamically, edit configs in YAML editor, run with overrides
2. **Browse Runs** — select a run, view results/config/summary, filter by suite/status/trainer, resume a run
3. **Job Inspector** — drill into individual jobs, view meta/events/TensorBoard dir

The `--share` flag generates a public URL (auto-detected on Colab).

**Note:** This cell blocks while the UI is running. Open the printed URL to interact.

In [ ]:
!PYTHONPATH=/content/OpenTrace:/content/Trace-Bench:$PYTHONPATH python -m trace_bench ui --runs-dir "$RUNS_DIR" --share


## TensorBoard

Each job stores TensorBoard event files under `jobs/<job_id>/tb/`. When the trainer uses `TensorboardLogger`, the runner auto-injects the per-job logdir.

To view:
```bash
tensorboard --logdir "$RUNS_DIR/<run_id>/jobs"
```

The Gradio UI also has a "Launch TensorBoard" button in the Browse tab.

## MLflow (optional)

MLflow mirroring is **opt-in**. Set `MLFLOW_TRACKING_URI` to enable:
```bash
export MLFLOW_TRACKING_URI=mlruns
python -m trace_bench run --config configs/m3_ui_demo.yaml --runs-dir runs
```

When enabled, the runner mirrors run params, job metrics, and artifact links to MLflow. The filesystem remains the canonical source of truth.

**Note:** Deep MLflow/OTel logger integration depends on the Trace team's upcoming PR. Current behavior is minimal mirroring of `score_initial`, `score_final`, `score_best`, and `time_seconds`.

## Summary

**M3 deliverables demonstrated:**
- Gradio UI with 3 tabs: Launch, Browse, Job Inspector
- Dynamic task/trainer discovery (non-hardcoded)
- Config editor: load, edit, save, run YAML configs from the UI
- Resume a previous run preserving the original run_id
- TensorBoard logdir auto-injection per job
- MLflow opt-in mirroring (filesystem canonical)
- CLI: `trace-bench ui --runs-dir ... --share --tasks-root ... --port ...`
- 22 M3 tests passing